In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='Set2')
print('Libraries loaded.')

## 1. Dataset Description & Loading
The **Mall Customers** dataset contains 200 customer records with:
- `CustomerID`, `Gender`, `Age`, `Annual Income (k$)`, `Spending Score (1-100)`

Download from Kaggle: https://www.kaggle.com/vjchoudhary7/customer-segmentation-tutorial-in-python

In [ ]:
# Load dataset
# Place Mall_Customers.csv in the working directory
df = pd.read_csv('Mall_Customers.csv')
print(f'Shape: {df.shape}')
df.head(10)

## 2. Data Cleaning & Preprocessing

In [ ]:
# Rename for convenience
df.rename(columns={
    'Annual Income (k$)': 'Income',
    'Spending Score (1-100)': 'SpendingScore'
}, inplace=True)

print('Data types:\n', df.dtypes)
print('\nMissing values:\n', df.isnull().sum())
print('\nBasic statistics:')
df.describe()

In [ ]:
# Encode Gender
df['Gender_enc'] = (df['Gender'] == 'Male').astype(int)

# Feature matrix for clustering
features = ['Age', 'Income', 'SpendingScore']
X = df[features].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print('Scaling complete. Sample:')
pd.DataFrame(X_scaled, columns=features).head(3)

## 3. Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Distributions
for ax, col, color in zip(axes[0], features, ['steelblue','darkorange','seagreen']):
    df[col].hist(bins=20, ax=ax, color=color, edgecolor='white')
    ax.set_title(f'{col} Distribution')
    ax.set_xlabel(col)

# Gender split
df['Gender'].value_counts().plot(kind='pie', ax=axes[1][0],
                                  autopct='%1.1f%%', colors=['#FF9999','#66B2FF'])
axes[1][0].set_title('Gender Split')
axes[1][0].set_ylabel('')

# Income vs Spending Score
axes[1][1].scatter(df['Income'], df['SpendingScore'], alpha=0.6, c='purple')
axes[1][1].set_xlabel('Annual Income (k$)')
axes[1][1].set_ylabel('Spending Score')
axes[1][1].set_title('Income vs Spending Score')

# Age vs Spending Score
axes[1][2].scatter(df['Age'], df['SpendingScore'], alpha=0.6, c='teal')
axes[1][2].set_xlabel('Age')
axes[1][2].set_ylabel('Spending Score')
axes[1][2].set_title('Age vs Spending Score')

plt.tight_layout()
plt.savefig('task2_eda.png', dpi=120)
plt.show()

## 4. K-Means Clustering — Optimal K via Elbow Method & Silhouette Score

In [ ]:
inertia = []
sil_scores = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertia.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(K_range, inertia, 'bo-', lw=2)
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia (WCSS)')
axes[0].set_title('Elbow Method')

axes[1].plot(K_range, sil_scores, 'rs-', lw=2)
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score vs K')

plt.tight_layout()
plt.savefig('task2_elbow.png', dpi=120)
plt.show()

best_k = K_range.start + np.argmax(sil_scores)
print(f'Best K by silhouette: {best_k}  (score={max(sil_scores):.3f})')

In [ ]:
# ── Fit final K-Means ─────────────────────────────────────────────────────────
K_FINAL = 5  # classic choice for mall customers; adjust based on elbow analysis
kmeans = KMeans(n_clusters=K_FINAL, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_scaled)

print('Cluster sizes:')
print(df['Cluster'].value_counts().sort_index())
print('\nCluster centroids (original scale):')
pd.DataFrame(scaler.inverse_transform(kmeans.cluster_centers_),
             columns=features).round(1)

## 5. Cluster Visualizations

In [ ]:
colors = ['#E63946','#457B9D','#2A9D8F','#E9C46A','#9B2226']
palette = {i: colors[i] for i in range(K_FINAL)}

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Income vs Spending Score with clusters
for c in range(K_FINAL):
    sub = df[df['Cluster'] == c]
    axes[0].scatter(sub['Income'], sub['SpendingScore'],
                    color=colors[c], label=f'Cluster {c}', s=60, edgecolors='white', lw=0.5)
cx = scaler.inverse_transform(kmeans.cluster_centers_)
axes[0].scatter(cx[:,1], cx[:,2], marker='X', s=200, c='black', zorder=5, label='Centroids')
axes[0].set_xlabel('Annual Income (k$)')
axes[0].set_ylabel('Spending Score')
axes[0].set_title('K-Means Clusters: Income vs Spending Score')
axes[0].legend()

# Age vs Spending Score with clusters
for c in range(K_FINAL):
    sub = df[df['Cluster'] == c]
    axes[1].scatter(sub['Age'], sub['SpendingScore'],
                    color=colors[c], label=f'Cluster {c}', s=60, edgecolors='white', lw=0.5)
axes[1].set_xlabel('Age')
axes[1].set_ylabel('Spending Score')
axes[1].set_title('K-Means Clusters: Age vs Spending Score')
axes[1].legend()

plt.tight_layout()
plt.savefig('task2_clusters.png', dpi=120)
plt.show()

In [ ]:
# ── PCA 2D Visualization ──────────────────────────────────────────────────────
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for c in range(K_FINAL):
    mask = df['Cluster'] == c
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    color=colors[c], label=f'Cluster {c}', s=60, edgecolors='white', lw=0.5)
axes[0].set_title(f'PCA Projection (explained var: {pca.explained_variance_ratio_.sum():.1%})')
axes[0].set_xlabel('PC 1')
axes[0].set_ylabel('PC 2')
axes[0].legend()

# ── t-SNE Visualization ───────────────────────────────────────────────────────
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
X_tsne = tsne.fit_transform(X_scaled)

for c in range(K_FINAL):
    mask = df['Cluster'] == c
    axes[1].scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                    color=colors[c], label=f'Cluster {c}', s=60, edgecolors='white', lw=0.5)
axes[1].set_title('t-SNE Projection')
axes[1].set_xlabel('t-SNE 1')
axes[1].set_ylabel('t-SNE 2')
axes[1].legend()

plt.tight_layout()
plt.savefig('task2_pca_tsne.png', dpi=120)
plt.show()

In [ ]:
# ── Cluster Profile Summary ───────────────────────────────────────────────────
profile = df.groupby('Cluster')[features].mean().round(1)
profile['Count'] = df.groupby('Cluster')['Age'].count()
print(profile)

# Radar / box plots per cluster
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, feat in zip(axes, features):
    df.boxplot(column=feat, by='Cluster', ax=ax,
               patch_artist=True,
               boxprops=dict(facecolor='lightblue'))
    ax.set_title(f'{feat} by Cluster')
    ax.set_xlabel('Cluster')
plt.suptitle('')
plt.tight_layout()
plt.savefig('task2_boxplots.png', dpi=120)
plt.show()